In [1]:
"""
=============================================================================
DCGAN — Synthetic Mammography Multi-View (CC + MLO)
Arsitektur : Deep Convolutional GAN (Radford et al., 2015)
Dataset    : VinDr-Mammo | Kelas: Density_A, Density_B, Density_D
Target    :
    Density_A → 2.000 pasangan sintetis  (asli: 40)
    Density_B → 4.000 pasangan sintetis  (asli: 764)
    Density_D → 4.000 pasangan sintetis  (asli: 1.080)
Optimasi  : Mixed Precision (AMP) + T4-friendly
=============================================================================
"""

# ===========================================================================
# 1. IMPORTS
# ===========================================================================

import os, glob, re, time
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# AMP API baru (torch.amp) + fallback utk torch versi lama
try:
    from torch.amp import GradScaler, autocast
except Exception:  # pragma: no cover
    from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as transforms
import torchvision.utils as vutils


# ===========================================================================
# 2. KONFIGURASI
# ===========================================================================

class BaseConfig:
    # ── Path ──────────────────────────────────────────────────────────────
    # NOTE: Path lokal Windows (ganti dari path Kaggle)
    DATA_ROOT   = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\dataset_density\train"

    # Output sintetis + model + samples disimpan ke folder lokal berikut
    OUTPUT_ROOT = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\Sintetis Density\synthetic_dataset\train"
    MODEL_DIR   = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\Sintetis Density\models"
    SAMPLE_DIR  = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\Sintetis Density\samples"

    # ── Resolusi ───────────────────────────────────────────────────────────
    IMG_SIZE    = 128     # Resolusi training. Naikkan ke 224 jika VRAM cukup

    # ── Arsitektur DCGAN ───────────────────────────────────────────────────
    Z_DIM   = 128    # Dimensi latent vector
    NGF     = 128    # Base feature map width — Generator
    NDF     = 128    # Base feature map width — Discriminator
    N_CHAN  = 2      # Output channel: 2 (CC + MLO sekaligus)

    # ── Optimizer ──────────────────────────────────────────────────────────
    LR_G    = 0.0002
    LR_D    = 0.0001
    BETA1   = 0.5    # Paper DCGAN: 0.5
    BETA2   = 0.999

    # ── Mixed Precision ────────────────────────────────────────────────────
    # AMP hanya aktif kalau CUDA tersedia.
    USE_AMP = True

    # ── Generasi sintetis ───────────────────────────────────────────────────
    GEN_BATCH_SIZE = 128

    # ── Misc ────────────────────────────────────────────────────────────────
    SEED            = 42

    # Windows sering crash kalau DataLoader pakai multiprocessing.
    # Set 0 supaya stabil (boleh dinaikkan kalau sudah aman di mesin kamu).
    NUM_WORKERS     = 0

    SAVE_INTERVAL   = 20
    SAMPLE_INTERVAL = 10

    # ── Label Smoothing ────────────────────────────────────────────────────
    REAL_LABEL_SMOOTH = 0.9  # Real labels = 0.9 (bukan 1.0)
    FAKE_LABEL        = 0.0

    # ── Model selection untuk generasi ────────────────────────────────────
    # True  → pakai generator epoch terakhir
    # False → pakai best model berdasarkan G_loss terendah
    USE_LAST_EPOCH = True


# Konfigurasi khusus per kelas density
CLASS_CONFIGS = {
    "Density_A": {
        "target_synth": 1000,
        "real_count":   40,
        # Data sangat sedikit → epoch lebih banyak + augmentasi agresif
        "epochs":       600,
        "batch_size":   32,    # Kecil agar tidak cepat overfit dengan 40 data
        "augment_extra": True,
    },
    "Density_B": {
        "target_synth": 4000,
        "real_count":   764,
        "epochs":       400,
        "batch_size":   64,
        "augment_extra": False,
    },
    "Density_D": {
        "target_synth": 4000,
        "real_count":   1080,
        "epochs":       400,
        "batch_size":   64,
        "augment_extra": False,
    },
}


# ===========================================================================
# 3. DATASET
# ===========================================================================

class MammographyPairDataset(Dataset):
    """
    Memuat pasangan CC + MLO dari satu kelas density.
    Output per item: (2, H, W) — grayscale, channel 0=CC, channel 1=MLO
    """
    def __init__(self, root, density_class, img_size, augment=True, augment_extra=False):
        self.img_size = img_size
        cc_dir  = os.path.join(root, density_class, "CC")
        mlo_dir = os.path.join(root, density_class, "MLO")

        cc_files  = {os.path.basename(f) for f in glob.glob(os.path.join(cc_dir,  "*.png"))}
        mlo_files = {os.path.basename(f) for f in glob.glob(os.path.join(mlo_dir, "*.png"))}
        common    = sorted(cc_files & mlo_files)

        self.pairs = [
            (os.path.join(cc_dir, fn), os.path.join(mlo_dir, fn))
            for fn in common
        ]
        print(f"[Dataset] {density_class} @ {img_size}px: {len(self.pairs)} pasangan")

        if len(self.pairs) == 0:
            raise RuntimeError(f"Tidak ada pasangan CC+MLO di {cc_dir} / {mlo_dir}")

        # ── Augmentasi ────────────────────────────────────────────────────
        aug_list = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=10),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ] if augment else []

        # Augmentasi ekstra untuk Density_A (data sangat sedikit: 40 pasangan)
        if augment_extra:
            aug_list += [
                transforms.RandomRotation(degrees=20),
                transforms.RandomAffine(degrees=5, translate=(0.08, 0.08), scale=(0.9, 1.1)),
                transforms.ColorJitter(brightness=0.3, contrast=0.3),
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
            ]

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            *aug_list,
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),   # [-1, 1]
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        cc_path, mlo_path = self.pairs[idx]
        cc  = Image.open(cc_path).convert("L")
        mlo = Image.open(mlo_path).convert("L")
        # Stack → (2, H, W)
        return torch.cat([self.transform(cc), self.transform(mlo)], dim=0)

    def get_max_id(self):
        ids = []
        for p, _ in self.pairs:
            m = re.search(r"breast_(\d+)\.png", os.path.basename(p))
            if m:
                ids.append(int(m.group(1)))
        return max(ids) if ids else 0


# ===========================================================================
# 4. MODEL — DCGAN
# ===========================================================================

def weights_init(m):
    """Inisialisasi bobot sesuai paper DCGAN (mean=0, std=0.02)."""
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ---------------------------------------------------------------------------
# 4a. Generator
#
#   Input : z (B, Z_DIM, 1, 1)
#   Output: (B, N_CHAN, IMG_SIZE, IMG_SIZE)
#
#   Untuk IMG_SIZE=64 : 4 blok (4→8→16→32→64)
#   Untuk IMG_SIZE=128: 5 blok (4→8→16→32→64→128)
#   Untuk IMG_SIZE=224: 5 blok + interpolate ke 224
# ---------------------------------------------------------------------------

class Generator(nn.Module):
    def __init__(self, z_dim: int, ngf: int, n_chan: int, img_size: int = 128):
        super().__init__()
        self.img_size = img_size

        # Hitung berapa blok upsampling yang dibutuhkan
        # Setiap blok: x2 resolusi, mulai dari 4×4
        n_blocks = max(4, int(np.log2(img_size)) - 1)

        layers = [
            # Blok pertama: z(B, Z_DIM, 1, 1) → (B, ngf*8, 4, 4)
            nn.ConvTranspose2d(z_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
        ]

        in_ch  = ngf * 8
        out_ch = ngf * 4
        for i in range(n_blocks - 1):
            layers += [
                nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(True),
            ]
            in_ch  = out_ch
            out_ch = max(out_ch // 2, ngf)

        # Head: channel terakhir → n_chan (CC + MLO)
        layers += [
            nn.ConvTranspose2d(in_ch, n_chan, 4, 2, 1, bias=False),
            nn.Tanh(),
        ]

        self.net = nn.Sequential(*layers)
        self.apply(weights_init)

    def forward(self, z):
        x = z.view(z.size(0), -1, 1, 1)
        out = self.net(x)
        # Resize jika resolusi output ≠ img_size (contoh: 224)
        if out.shape[-1] != self.img_size:
            out = nn.functional.interpolate(
                out, size=self.img_size, mode="bilinear", align_corners=False
            )
        return out


# ---------------------------------------------------------------------------
# 4b. Discriminator
#
#   Input : (B, N_CHAN, IMG_SIZE, IMG_SIZE)
#   Output: (B,) — logit skalar per sampel
#
#   Input selalu di-resize ke 64 agar conv terakhir menghasilkan (B,1,1,1)
# ---------------------------------------------------------------------------

class Discriminator(nn.Module):
    def __init__(self, ndf: int, n_chan: int):
        super().__init__()

        self.net = nn.Sequential(
            # Blok pertama: tidak pakai BN (sesuai paper DCGAN)
            nn.Conv2d(n_chan, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf,     ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Output: 64×64 → 4×4 → (B, 1, 1, 1) → flatten → (B, 1)
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )
        self.apply(weights_init)

    def forward(self, x):
        # Selalu resize ke 64 agar conv terakhir konsisten
        if x.shape[-1] != 64:
            x = nn.functional.interpolate(x, size=64, mode="bilinear", align_corners=False)
        return self.net(x).view(-1)


# ===========================================================================
# 5. TRAINING DCGAN — per kelas
# ===========================================================================

def train_dcgan_class(
    density_class: str,
    generator: Generator,
    discriminator: Discriminator,
    cfg: BaseConfig,
    class_cfg: dict,
    device: torch.device,
) -> Generator:
    """Training DCGAN lengkap untuk satu kelas density."""

    img_size      = cfg.IMG_SIZE
    epochs        = class_cfg["epochs"]
    batch_size    = class_cfg["batch_size"]
    augment_extra = class_cfg["augment_extra"]

    dataset = MammographyPairDataset(
        cfg.DATA_ROOT, density_class, img_size,
        augment=True, augment_extra=augment_extra
    )

    # Guard: Windows multiprocessing DataLoader sering "worker exited unexpectedly"
    safe_num_workers = int(cfg.NUM_WORKERS)
    if os.name == "nt" and safe_num_workers != 0:
        print("[WARN] Windows detected: override NUM_WORKERS -> 0 (stabil)")
        safe_num_workers = 0

    loader = DataLoader(
        dataset,
        batch_size  = batch_size,
        shuffle     = True,
        num_workers = safe_num_workers,
        pin_memory  = device.type == "cuda",
        drop_last   = True,
    )

    criterion = nn.BCEWithLogitsLoss()

    opt_g = optim.Adam(generator.parameters(),     lr=cfg.LR_G, betas=(cfg.BETA1, cfg.BETA2))
    opt_d = optim.Adam(discriminator.parameters(), lr=cfg.LR_D, betas=(cfg.BETA1, cfg.BETA2))

    # AMP hanya untuk CUDA
    amp_enabled = bool(cfg.USE_AMP and device.type == "cuda")

    # torch.amp.GradScaler(device) pada versi baru
    try:
        scaler_g = GradScaler("cuda", enabled=amp_enabled)
        scaler_d = GradScaler("cuda", enabled=amp_enabled)
    except TypeError:
        # fallback untuk torch versi lama
        scaler_g = GradScaler(enabled=amp_enabled)
        scaler_d = GradScaler(enabled=amp_enabled)

    fixed_noise = torch.randn(16, cfg.Z_DIM, device=device)
    best_g_loss = float("inf")

    best_path = os.path.join(cfg.MODEL_DIR, f"{density_class}_dcgan_generator_best.pth")
    last_path  = os.path.join(cfg.MODEL_DIR, f"{density_class}_dcgan_generator_last.pth")

    sample_dir = os.path.join(cfg.SAMPLE_DIR, density_class)
    os.makedirs(sample_dir, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  [{density_class}] DCGAN Training | {img_size}×{img_size} | {epochs} epoch | batch {batch_size}")
    print(f"  Real data      : {class_cfg['real_count']} pasangan")
    print(f"  Target sintetis: {class_cfg['target_synth']} pasangan")
    print(f"  Augment extra  : {augment_extra}")
    print(f"  AMP            : {amp_enabled}")
    print(f"  num_workers    : {safe_num_workers}")
    print(f"  Model output   : {'epoch terakhir' if cfg.USE_LAST_EPOCH else 'best G_loss'}")
    print(f"{'='*65}")

    for epoch in range(1, epochs + 1):
        generator.train()
        discriminator.train()

        d_loss_sum = g_loss_sum = 0.0
        t0 = time.time()

        for real_imgs in loader:
            # real_imgs: (B, 2, H, W)
            real_imgs = real_imgs.to(device)
            B = real_imgs.size(0)

            real_labels = torch.full((B,), cfg.REAL_LABEL_SMOOTH, device=device)
            fake_labels = torch.full((B,), cfg.FAKE_LABEL,        device=device)

            # ── Update Discriminator ───────────────────────────────────
            opt_d.zero_grad()
            with autocast(device_type="cuda", enabled=amp_enabled):
                d_real   = discriminator(real_imgs)
                loss_d_r = criterion(d_real, real_labels)

                z      = torch.randn(B, cfg.Z_DIM, device=device)
                fake   = generator(z).detach()
                d_fake = discriminator(fake)
                loss_d_f = criterion(d_fake, fake_labels)

                loss_d = (loss_d_r + loss_d_f) * 0.5

            scaler_d.scale(loss_d).backward()
            scaler_d.unscale_(opt_d)
            nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
            scaler_d.step(opt_d)
            scaler_d.update()

            # ── Update Generator ───────────────────────────────────────
            opt_g.zero_grad()
            with autocast(device_type="cuda", enabled=amp_enabled):
                z      = torch.randn(B, cfg.Z_DIM, device=device)
                fake   = generator(z)
                d_fake = discriminator(fake)
                loss_g = criterion(d_fake, real_labels)  # G ingin D percaya fake = real

            scaler_g.scale(loss_g).backward()
            scaler_g.unscale_(opt_g)
            nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
            scaler_g.step(opt_g)
            scaler_g.update()

            d_loss_sum += loss_d.item()
            g_loss_sum += loss_g.item()

        n = len(loader)
        avg_d = d_loss_sum / max(n, 1)
        avg_g = g_loss_sum / max(n, 1)
        elapsed = time.time() - t0
        print(f"  Epoch {epoch:>3}/{epochs}  D:{avg_d:.4f}  G:{avg_g:.4f}  {elapsed:.1f}s")

        # Simpan best model
        if avg_g < best_g_loss:
            best_g_loss = avg_g
            torch.save({
                "generator":     generator.state_dict(),
                "discriminator": discriminator.state_dict(),
                "g_loss":        best_g_loss,
                "epoch":         epoch,
                "density_class": density_class,
                "img_size":      img_size,
            }, best_path)
            print(f"  ✓ Best model disimpan (G_loss={best_g_loss:.4f}, epoch={epoch})")

        # Simpan sampel gambar berkala
        if epoch % cfg.SAMPLE_INTERVAL == 0:
            generator.eval()
            with torch.no_grad():
                with autocast(device_type="cuda", enabled=amp_enabled):
                    fake_sample = generator(fixed_noise)

            vutils.save_image(
                fake_sample[:, 0:1],
                os.path.join(sample_dir, f"cc_ep{epoch}.png"),
                normalize=True, nrow=4,
            )
            vutils.save_image(
                fake_sample[:, 1:2],
                os.path.join(sample_dir, f"mlo_ep{epoch}.png"),
                normalize=True, nrow=4,
            )

    # Simpan state epoch terakhir
    torch.save({
        "generator":     generator.state_dict(),
        "discriminator": discriminator.state_dict(),
        "epoch":         epochs,
        "density_class": density_class,
        "img_size":      img_size,
    }, last_path)
    print(f"\n  ✓ Last epoch model disimpan → {last_path}")

    # Pilih model untuk generasi
    if cfg.USE_LAST_EPOCH:
        print(f"[INFO] [{density_class}] USE_LAST_EPOCH=True → memakai epoch terakhir ({epochs})")
    else:
        ckpt = torch.load(best_path, map_location=device)
        generator.load_state_dict(ckpt["generator"])
        print(f"[INFO] [{density_class}] Best model dimuat "
              f"(epoch {ckpt['epoch']}, G_loss={ckpt['g_loss']:.4f})")

    return generator


# ===========================================================================
# 6. GENERASI DATA SINTETIS
# ===========================================================================

def tensor_to_pil(t: torch.Tensor) -> Image.Image:
    """Konversi tensor (1, H, W) dari [-1,1] ke PIL Image grayscale."""
    t = t.squeeze(0).cpu().float()
    t = ((t + 1.0) / 2.0).clamp(0, 1)
    return Image.fromarray((t.numpy() * 255).astype(np.uint8), mode="L")


def generate_synthetic_pairs(
    generator: Generator,
    density_class: str,
    start_id: int,
    n_pairs: int,
    cfg: BaseConfig,
    device: torch.device,
):
    out_cc  = os.path.join(cfg.OUTPUT_ROOT, density_class, "CC")
    out_mlo = os.path.join(cfg.OUTPUT_ROOT, density_class, "MLO")
    os.makedirs(out_cc,  exist_ok=True)
    os.makedirs(out_mlo, exist_ok=True)

    generator.eval()
    generated  = 0
    current_id = start_id
    t0         = time.time()

    print(f"\n[INFO] [{density_class}] Generating {n_pairs} pasangan @ {cfg.IMG_SIZE}×{cfg.IMG_SIZE}...")
    print(f"[INFO] File ID mulai: breast_{start_id}.png")

    with torch.no_grad():
        while generated < n_pairs:
            this_batch = min(cfg.GEN_BATCH_SIZE, n_pairs - generated)
            z    = torch.randn(this_batch, cfg.Z_DIM, device=device)

            with autocast(enabled=cfg.USE_AMP):
                fake = generator(z)            # (B, 2, H, W)

            cc_batch  = fake[:, 0:1]           # (B, 1, H, W)
            mlo_batch = fake[:, 1:2]           # (B, 1, H, W)

            for i in range(this_batch):
                fname = f"breast_{current_id}.png"
                tensor_to_pil(cc_batch[i]).save(os.path.join(out_cc,  fname))
                tensor_to_pil(mlo_batch[i]).save(os.path.join(out_mlo, fname))
                current_id += 1

            generated += this_batch

            if generated % 500 == 0 or generated == n_pairs:
                elapsed = time.time() - t0
                rate    = generated / max(elapsed, 1e-6)
                eta     = (n_pairs - generated) / max(rate, 1e-6)
                print(f"  [{generated:>6}/{n_pairs}]  {rate:.0f} pair/s  ETA: {eta:.0f}s")

    print(f"[INFO] [{density_class}] Selesai. ID: breast_{start_id} → breast_{current_id-1}")


# ===========================================================================
# 7. VERIFIKASI
# ===========================================================================

def verify_class_output(density_class: str, cfg: BaseConfig, class_cfg: dict):
    out_cc  = os.path.join(cfg.OUTPUT_ROOT, density_class, "CC")
    out_mlo = os.path.join(cfg.OUTPUT_ROOT, density_class, "MLO")

    cc_files  = {os.path.basename(f) for f in glob.glob(os.path.join(out_cc,  "*.png"))}
    mlo_files = {os.path.basename(f) for f in glob.glob(os.path.join(out_mlo, "*.png"))}

    missing_mlo = cc_files - mlo_files
    missing_cc  = mlo_files - cc_files
    total       = class_cfg["real_count"] + len(cc_files)

    print(f"\n  [{density_class}]")
    print(f"    CC  sintetis : {len(cc_files)}")
    print(f"    MLO sintetis : {len(mlo_files)}")
    if not missing_mlo and not missing_cc:
        print(f"    ✓ Semua pasangan simetris dan lengkap!")
    else:
        if missing_mlo: print(f"    ⚠ CC tanpa MLO : {len(missing_mlo)}")
        if missing_cc:  print(f"    ⚠ MLO tanpa CC : {len(missing_cc)}")
    print(f"    Total akhir  : {class_cfg['real_count']} (asli) + {len(cc_files)} (sintetis) = {total}")


def verify_all(cfg: BaseConfig):
    print(f"\n{'='*65}")
    print(f"  VERIFIKASI OUTPUT SEMUA KELAS")
    print(f"{'='*65}")
    for cls, ccfg in CLASS_CONFIGS.items():
        verify_class_output(cls, cfg, ccfg)

    print(f"\n  Distribusi Akhir (asli + sintetis):")
    print(f"    {'Kelas':<12} {'Asli':>8} {'Sintetis':>10} {'Total':>8}")
    print(f"    {'-'*42}")
    for cls, ccfg in CLASS_CONFIGS.items():
        out_cc = os.path.join(cfg.OUTPUT_ROOT, cls, "CC")
        synth  = len(glob.glob(os.path.join(out_cc, "*.png")))
        total  = ccfg["real_count"] + synth
        print(f"    {cls:<12} {ccfg['real_count']:>8} {synth:>10} {total:>8}")
    print(f"    {'Density_C':<12} {'6115':>8} {'0':>10} {'6115':>8}  (tidak diaugment)")
    print(f"{'='*65}")


# ===========================================================================
# 8. MAIN
# ===========================================================================

def main():
    cfg = BaseConfig()

    torch.manual_seed(cfg.SEED)
    np.random.seed(cfg.SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[INFO] Device : {device}")
    if device.type == "cuda":
        print(f"[INFO] GPU    : {torch.cuda.get_device_name(0)}")
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"[INFO] VRAM   : {vram:.1f} GB")
        if vram < 8:
            print("[WARN] VRAM < 8GB — batch_size Density_A sudah 16 (aman)")

    os.makedirs(cfg.MODEL_DIR,  exist_ok=True)
    os.makedirs(cfg.SAMPLE_DIR, exist_ok=True)

    print("\n" + "="*65)
    print("  DCGAN — VinDr-Mammo Breast Density Synthetic Generation")
    print("  Arsitektur: Radford et al. (2015) — tanpa pretrained")
    print("  Kelas: Density_A, Density_B, Density_D")
    print("="*65)

    for density_class in ["Density_A", "Density_B", "Density_D"]:
        class_cfg = CLASS_CONFIGS[density_class]

        print(f"\n{'#'*65}")
        print(f"  PROSES KELAS : {density_class}")
        print(f"{'#'*65}")

        # Cari max ID dataset asli agar ID sintetis tidak bertabrakan
        dummy_ds = MammographyPairDataset(
            cfg.DATA_ROOT, density_class, 64,
            augment=False, augment_extra=False
        )
        max_id   = dummy_ds.get_max_id()
        start_id = max_id + 1
        del dummy_ds
        print(f"[INFO] [{density_class}] Max ID asli: breast_{max_id} → sintetis mulai breast_{start_id}")

        # Inisialisasi DCGAN
        generator     = Generator(cfg.Z_DIM, cfg.NGF, cfg.N_CHAN, cfg.IMG_SIZE).to(device)
        discriminator = Discriminator(cfg.NDF, cfg.N_CHAN).to(device)

        n_g = sum(p.numel() for p in generator.parameters())
        n_d = sum(p.numel() for p in discriminator.parameters())
        print(f"[INFO] Generator params     : {n_g:,}")
        print(f"[INFO] Discriminator params : {n_d:,}")

        # Training
        generator = train_dcgan_class(
            density_class, generator, discriminator, cfg, class_cfg, device
        )

        # Bebaskan VRAM discriminator sebelum generate
        del discriminator
        torch.cuda.empty_cache()

        # Generate data sintetis
        generate_synthetic_pairs(
            generator, density_class,
            start_id = start_id,
            n_pairs  = class_cfg["target_synth"],
            cfg      = cfg,
            device   = device,
        )

        # Bebaskan VRAM sebelum kelas berikutnya
        del generator
        torch.cuda.empty_cache()
        print(f"\n[INFO] [{density_class}] Selesai. VRAM dibebaskan.\n")

    # Verifikasi semua output
    verify_all(cfg)

    print("\n[DONE] Pipeline selesai!")
    print(f"       Output  : {cfg.OUTPUT_ROOT}")
    print(f"       Model   : {cfg.MODEL_DIR}/{{kelas}}_dcgan_generator_best.pth")
    print(f"       Samples : {cfg.SAMPLE_DIR}/{{kelas}}/")


if __name__ == "__main__":
    main()

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\lib\shm.dll" or one of its dependencies.